# Investment Backtest Lab Prototype

Load the MVP config, run an offline strategy smoke test, then switch to live adapters once the environment and tokens are ready.

In [ ]:
from investment_backtest_lab.config import load_backtest_config
from investment_backtest_lab.costs import CostModel

config = load_backtest_config('../configs/mvp_example.yaml')
cost_model = CostModel.from_dict(config.cost_model)
config

In [ ]:
import numpy as np
import pandas as pd
from investment_backtest_lab.reports import performance_summary
from investment_backtest_lab.strategies.vectorbt_wrappers import moving_average_signals, position_from_signals

index = pd.bdate_range(config.start_date, config.end_date)
rng = np.random.default_rng(42)
returns = pd.Series(0.00025 + rng.normal(0, 0.01, len(index)), index=index)
close = (100 * (1 + returns).cumprod()).rename('SYNTH')
ma_params = {
    'fast_window': config.strategy.params.get('fast_window', 50),
    'slow_window': config.strategy.params.get('slow_window', 200),
}
entries, exits = moving_average_signals(close, **ma_params)
position = position_from_signals(entries, exits)
strategy_returns = close.pct_change().where(position.shift().fillna(False), 0).fillna(0)
performance_summary(strategy_returns)

In [ ]:
from investment_backtest_lab.data import MarketDataLoader

# Uncomment after Python, uv, dependencies, network, and FINMIND_TOKEN are ready.
# loader = MarketDataLoader(use_cache=True)
# close_prices = loader.load_close_prices(
#     config.universe,
#     start_date=config.start_date.isoformat(),
#     end_date=config.end_date.isoformat(),
# )
# close_prices.tail()